# Phase 3c: Per-date field means from Google Earth Engine

The Phase 3 rasters are **monthly** composites, so each field has only 6 time steps (June to November). That is too short for the deep learning models in Phase 6 (LSTM, InceptionTime, Transformer). This notebook exports, for every field, the value of **every satellite pass**:

| Part | What | Steps per season | Output (Google Drive, folder `thesis_perdate`) |
|---|---|---|---|
| 1 | Sentinel-1 VV and VH, orbit 63, Lee sigma filter, field mean in linear power | 14 to 15 (every 12 days) | `S1_perdate_<year>.csv` |
| 2 | Sentinel-2 NDVI, EVI, GCVI, LSWI, NDRE, mean of the clear pixels, plus the share of clear pixels | up to about 35 (every 5 days), many cloudy | `S2_perdate_<year>.csv` |

**Same settings as Phase 3 and 4**, so nothing new is chosen here: orbit 63 descending (P3a Step 3), Lee sigma 5 × 5 (P3b), Cloud Score+ 0.60 (P3a), the same index formulas (P3a), and each field shrunk by 10 m inside (Phase 4). The output is tables only (no rasters), about 8 small CSV files. Only `uid`, `year` and the outlines are sent to Earth Engine, as in P3a.

Run it like P3a: run all cells, then re-run Step 3 from time to time until everything is `COMPLETED`, and download the folder `thesis_perdate` into `data/raster/perdate/`.

## Step 0: Setup and fields

Same login and project as P3a. The 1,205 fields are shrunk by 10 m inside, as in Phase 4; fields too small for that (less than 3 pixels left) keep their full outline.

**Found:** _(fill after running)_

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import geopandas as gpd
import ee

here = Path.cwd()
ROOT = here if (here / "data" / "raw").exists() else here.parent
DATA2 = ROOT / "data" / "processed_v2"
OUT = ROOT / "results" / "phase3"; OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / "src"))
import s1_speckle as sf                       # published speckle filters (Mullissa et al. 2021), as in P3a

PROJECT = "ee-josaiahsyiem"
DRIVE_FOLDER = "thesis_perdate"               # new Google Drive folder for these tables
CRS = "EPSG:32643"
YEARS = [2022, 2023, 2024, 2025]
PASS, ORBIT = "DESCENDING", 63                # the only orbit over Latur (P3a Step 3)
FILTER, KERNEL = "leesigma", 5                # the filter chosen in P3b, 5 x 5 window (as in P3a)
INNER_M = 10                                  # inner shrink, as in Phase 4 (leave out mixed border pixels)
CS_MIN = 0.60                                 # Cloud Score+ threshold, as in P3a

try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)
print("Earth Engine ready, project:", PROJECT)

# fields, shrunk by 10 m inside (fields too small for that keep their full outline)
fields = gpd.read_file(DATA2 / "yield_analysis_ready_v2.gpkg")[["uid", "year", "geometry"]].to_crs(CRS)
inner = fields.geometry.buffer(-INNER_M)
small = inner.is_empty | (inner.area < 300)                         # less than 3 pixels left
fields["geometry"] = inner.where(~small, fields.geometry).simplify(1)
fields["shrunk"] = (~small).astype(int)
fc_fields = ee.FeatureCollection(json.loads(fields.to_crs(4326).to_json(drop_id=True)))
print("fields sent:", fc_fields.size().getInfo(), "| per year:", fields["year"].value_counts().sort_index().to_dict(),
      "| kept full outline (too small to shrink):", int(small.sum()))
xmin, ymin, xmax, ymax = fields.total_bounds
REGION = ee.Geometry.Rectangle([xmin - 1000, ymin - 1000, xmax + 1000, ymax + 1000], proj=CRS, geodesic=False)

TASKS = []
def export_table(fc, name, columns):
    t = ee.batch.Export.table.toDrive(collection=fc, description=name, folder=DRIVE_FOLDER,
                                      fileNamePrefix=name, fileFormat="CSV", selectors=columns)
    t.start(); TASKS.append((name, t))

## Step 1: Sentinel-1, every pass

Every June to November image of orbit 63, Lee sigma filtered (as in P3a Part B). For every field of that year: the mean VV and VH in linear power. They are turned into dB in Phase 6, the same way as the monthly values (mean in linear power, then dB). One table per year; `tileScale = 8`, which avoided the memory error in P3a.

**Found:** _(fill after running)_

In [ ]:
# Sentinel-1: every image of orbit 63, Lee sigma filter, field mean in linear power (turned to dB later, as in P3a)
def s1_season(year):
    c = (ee.ImageCollection("COPERNICUS/S1_GRD_FLOAT")
         .filterBounds(REGION).filterDate(f"{year}-06-01", f"{year}-12-01")
         .filter(ee.Filter.eq("instrumentMode", "IW")).filter(ee.Filter.eq("resolution_meters", 10))
         .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
         .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
         .filter(ee.Filter.eq("orbitProperties_pass", PASS)).filter(ee.Filter.eq("relativeOrbitNumber_start", ORBIT))
         .select(["VV", "VH", "angle"])
         .map(lambda im: im.updateMask(im.select("VV").gt(1e-4))))
    return ee.ImageCollection(sf.MonoTemporal_Filter(c, KERNEL, "LEE SIGMA")).select(["VV", "VH"])

n0 = len(TASKS)
for yr in YEARS:
    f_y = fc_fields.filter(ee.Filter.eq("year", yr))
    coll = s1_season(yr)
    print(yr, "Sentinel-1 images:", coll.size().getInfo())
    def per_image(im):
        d = ee.Date(im.get("system:time_start")).format("YYYY-MM-dd")
        return (im.reduceRegions(collection=f_y, reducer=ee.Reducer.mean(), scale=10, crs=CRS, tileScale=8)
                .map(lambda ft: ft.set("date", d).setGeometry(None)))
    export_table(coll.map(per_image).flatten(), f"S1_perdate_{yr}", ["uid", "year", "date", "VV", "VH"])
print("Sentinel-1 exports started:", len(TASKS) - n0)

## Step 2: Sentinel-2, every pass

Every June to November image (all tiles), masked with Cloud Score+ as in P3a. For every field: the mean NDVI, EVI, GCVI, LSWI and NDRE over the **clear** pixels, and `clear` = the share of the field's pixels that are clear (0 to 1). Phase 6 keeps a field-date only if at least half its pixels are clear (the Phase 4 rule) and merges dates seen by two tiles.

**Found:** _(fill after running)_

In [ ]:
# Sentinel-2: every image, Cloud Score+ mask; index means over clear pixels, and the share of clear pixels
CS = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
def s2_prep(im):
    clear = im.select("cs_cdf").gte(CS_MIN)
    r = im.select(["B2", "B3", "B4", "B5", "B8", "B11"]).divide(10000)
    b2, b3, b4, b5, b8, b11 = [r.select(b) for b in ["B2", "B3", "B4", "B5", "B8", "B11"]]
    ndvi = b8.subtract(b4).divide(b8.add(b4)).rename("NDVI")
    evi = b8.subtract(b4).multiply(2.5).divide(b8.add(b4.multiply(6)).subtract(b2.multiply(7.5)).add(1)).rename("EVI")
    gcvi = b8.divide(b3).subtract(1).rename("GCVI")
    lswi = b8.subtract(b11).divide(b8.add(b11)).rename("LSWI")
    ndre = b8.subtract(b5).divide(b8.add(b5)).rename("NDRE")
    idx = ee.Image.cat([ndvi, evi, gcvi, lswi, ndre]).updateMask(clear)       # same formulas as P3a
    return idx.addBands(clear.rename("clear")).set("system:time_start", im.get("system:time_start"))

n0 = len(TASKS)
for yr in YEARS:
    f_y = fc_fields.filter(ee.Filter.eq("year", yr))
    coll = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(REGION)
            .filterDate(f"{yr}-06-01", f"{yr}-12-01").linkCollection(CS, ["cs_cdf"]).map(s2_prep))
    print(yr, "Sentinel-2 images (all tiles):", coll.size().getInfo())
    def per_image(im):
        d = ee.Date(im.get("system:time_start")).format("YYYY-MM-dd")
        return (im.reduceRegions(collection=f_y.filterBounds(im.geometry()), reducer=ee.Reducer.mean(), scale=10, crs=CRS, tileScale=8)
                .map(lambda ft: ft.set("date", d).setGeometry(None)))
    export_table(coll.map(per_image).flatten(), f"S2_perdate_{yr}", ["uid", "year", "date", "clear", "NDVI", "EVI", "GCVI", "LSWI", "NDRE"])
print("Sentinel-2 exports started:", len(TASKS) - n0)

## Step 3: Task status

Re-run this cell from time to time. When all 8 say `COMPLETED`, download the folder `thesis_perdate` from Google Drive into `data/raster/perdate/`. You can also follow the tasks at https://code.earthengine.google.com/tasks.

**Found:** _(fill after running)_

In [ ]:
def status():
    st = pd.DataFrame([{"file": n, "state": t.status()["state"], "error": t.status().get("error_message", "")} for n, t in TASKS])
    print(st["state"].value_counts().to_dict())
    bad = st[st["state"].isin(["FAILED", "CANCELLED"])]
    if len(bad): print(bad.to_string(index=False))
    return st
st = status()